<a href="https://colab.research.google.com/github/akashkumarcs24-cloud/aiLAB/blob/main/KnowledgeBase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from re import S
import re

class Symbol:
    def __init__(self, name):
        self.name = name
    def evaluate(self, model):
        return model.get(self.name, False)
    def __repr__(self):
        return self.name

class Not:
    def __init__(self, op):
        self.op = op
    def evaluate(self, model):
        return not self.op.evaluate(model)
    def __repr__(self):
        return f"~({self.op})"

class And:
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)
    def __repr__(self):
        return f"({self.left} & {self.right})"

class Or:
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)
    def __repr__(self):
        return f"({self.left} | {self.right})"

class Implies:
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def evaluate(self, model):
        return (not self.left.evaluate(model)) or self.right.evaluate(model)
    def __repr__(self):
        return f"({self.left} => {self.right})"


def get_symbols(expression):
    if isinstance(expression, Symbol):
        return {expression.name}
    elif isinstance(expression, Not):
        return get_symbols(expression.op)
    elif isinstance(expression, (And, Or, Implies)):
        return get_symbols(expression.left).union(get_symbols(expression.right))
    return set()

def evaluate_kb(kb, model):
    for clause in kb:
        if not clause.evaluate(model):
            return False
    return True

def generate_models(symbols, current_model={}):
    if not symbols:
        return [current_model]
    symbol = symbols[0]
    remaining = symbols[1:]
    return (generate_models(remaining, {**current_model, symbol: True}) +
            generate_models(remaining, {**current_model, symbol: False}))

def tt_entails_with_table(kb, query):
    symbols = set()
    for clause in kb:
        symbols.update(get_symbols(clause))
    symbols.update(get_symbols(query))
    symbol_list = sorted(list(symbols))

    models = generate_models(symbol_list, {})

    # Prepare column headers
    headers = list(symbol_list) + [f"KB_{i+1}: {clause}" for i, clause in enumerate(kb)] + ["KB_All", f"Query: {query}", "Entailed?"]
    row_format = " | ".join(["{:^10}"] * len(headers))

    print("\n" + "=" * (13 * len(headers)))
    print(row_format.format(*headers))
    print("=" * (13 * len(headers)))

    entails = True
    for model in models:
        model_vals = ["T" if model[sym] else "F" for sym in symbol_list]

        kb_vals = []
        for clause in kb:
            kb_vals.append("T" if clause.evaluate(model) else "F")

        kb_all_bool = evaluate_kb(kb, model)
        kb_all = "T" if kb_all_bool else "F"

        query_bool = query.evaluate(model)
        query_val = "T" if query_bool else "F"

        entailed_val = "-"
        if kb_all_bool:
            if query_bool:
                entailed_val = "YES"
            else:
                entailed_val = "NO"
                entails = False

        row_data = model_vals + kb_vals + [kb_all, query_val, entailed_val]
        print(row_format.format(*row_data))

    print("=" * (13 * len(headers)))
    return entails

# --- Parse logic string to logical objects ---

def parse_formula(formula_str):
    formula_str = formula_str.strip()

    # Handle parentheses
    if formula_str.startswith('(') and formula_str.endswith(')'):
        # Verify if outer parentheses match
        count = 0
        mismatch = False
        for i, char in enumerate(formula_str):
            if char == '(':
                count += 1
            elif char == ')':
                count -= 1
            if count == 0 and i < len(formula_str) - 1:
                mismatch = True
                break
        if not mismatch:
            return parse_formula(formula_str[1:-1])

    # Implication (->)
    parts = split_by_operator(formula_str, ['->'])
    if parts:
        return Implies(parse_formula(parts[0]), parse_formula(parts[1]))

    # Disjunction (v or |)
    parts = split_by_operator(formula_str, ['v', '|'])
    if parts:
        return Or(parse_formula(parts[0]), parse_formula(parts[1]))

    # Conjunction (^ or &)
    parts = split_by_operator(formula_str, ['^', '&'])
    if parts:
        return And(parse_formula(parts[0]), parse_formula(parts[1]))

    # Negation (~ or not)
    if formula_str.startswith('~'):
        return Not(parse_formula(formula_str[1:]))
    if formula_str.lower().startswith('not '):
        return Not(parse_formula(formula_str[4:]))

    # Base Symbol
    return Symbol(formula_str.strip())

def split_by_operator(s, ops):
    count = 0
    for i in range(len(s) - 1, -1, -1):
        char = s[i]
        if char == ')':
            count += 1
        elif char == '(':
            count -= 1
        elif count == 0:
            for op in ops:
                op_len = len(op)
                if i - op_len + 1 >= 0 and s[i - op_len + 1:i + 1].lower() == op:
                    return [s[:i - op_len + 1], s[i + 1:]]
    return None

# --- User Input Logic ---
print("Enter your Knowledge Base formulas (one per line, leave empty to finish).")
print("Supported operators:")
print("  Conjunction: ^  (or &)")
print("  Disjunction: v  (or |)")
print("  Negation:    ~")
print("  Implication: ->")
print("Example: P -> ~Q")
print("-" * 40)

kb_inputs = []
while True:
    inp = input("Enter KB formula: ").strip()
    if not inp:
        break
    kb_inputs.append(parse_formula(inp))

query_inp = input("Enter Query formula: ").strip()
if kb_inputs and query_inp:
    query_parsed = parse_formula(query_inp)
    result = tt_entails_with_table(kb_inputs, query_parsed)
    print(f"\nDoes KB entail the query '{query_parsed}'? {result}")
else:
    print("Error: KB and Query cannot be empty.")

Enter your Knowledge Base formulas (one per line, leave empty to finish).
Supported operators:
  Conjunction: ^  (or &)
  Disjunction: v  (or |)
  Negation:    ~
  Implication: ->
Example: P -> ~Q
----------------------------------------
Enter KB formula: Q -> P
Enter KB formula: P -> ~Q
Enter KB formula: Q v R
Enter KB formula: 
Enter Query formula: R -> P

    P      |     Q      |     R      | KB_1: (Q => P) | KB_2: (P => ~(Q)) | KB_3: (Q | R) |   KB_All   | Query: (R => P) | Entailed? 
    T      |     T      |     T      |     T      |     F      |     T      |     F      |     T      |     -     
    T      |     T      |     F      |     T      |     F      |     T      |     F      |     T      |     -     
    T      |     F      |     T      |     T      |     T      |     T      |     T      |     T      |    YES    
    T      |     F      |     F      |     T      |     T      |     F      |     F      |     T      |     -     
    F      |     T      |     T      |     F 